# AlphaFold2 Structural Features (Colab)

> يستخرج pLDDT + SASA + residue depth لكل موقع missense في بياناتنا.

## ماذا نحسب؟

لكل (UniProt ID, protein_position) في الـsplits:
- `pLDDT_position` — AlphaFold confidence في هذا الموقع (0–100)
- `pLDDT_window_5` — متوسط pLDDT في نافذة ±٥ بقايا
- `SASA_position` — المساحة المُعرّضة للماء (Ų) — المواقع المدفونة أكثر حساسية
- `secondary_structure` — DSSP 8-class (H, E, T, …)
- `neighbors_5A` — عدد CA في نطاق ٥ Ų

## مدخلات

نتايج ESM-2 (من notebook 11) تعطينا `transcript_id` + `protein_position` لكل variant. نحتاج نحوّل transcript → UniProt، ثم نحمّل PDB من AlphaFold DB.

## مدة التنفيذ

~٣-٥ ساعات على T4 (معظم الوقت download + DSSP).

---

## ١. التثبيت

In [ ]:
# Structural tools
!apt-get update -qq && apt-get install -y -qq dssp
%pip install -q biopython freesasa

import Bio, freesasa
print('Biopython:', Bio.__version__)

## ٢. Mount Drive + Clone repo

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

PDB_CACHE = '/content/drive/MyDrive/missense_alphafold/pdbs'
FEATURES_DIR = '/content/drive/MyDrive/missense_alphafold/features'
!mkdir -p {PDB_CACHE} {FEATURES_DIR}

GH_PAT = userdata.get('GH_PAT')
!rm -rf /content/repo
!git clone https://{GH_PAT}@github.com/RayanAlDwlah/Genetic-Mutation-Detection-project.git /content/repo
%cd /content/repo

## ٣. استخراج (uniprot, protein_position) لكل variant

نحتاج نتايج ESM-2 كمصدر للـtranscript_id. ثم نحوّلها إلى UniProt ID عبر VEP REST.

In [ ]:
import pandas as pd

# Load ESM-2 outputs from Notebook 11
esm_frames = []
for sp in ['train', 'val', 'test']:
    p = f'/content/repo/data/intermediate/esm2/scores_{sp}.parquet'
    try:
        esm_frames.append(pd.read_parquet(p))
    except FileNotFoundError:
        print(f'⚠ skip {sp} — {p} not found. Run notebook 11 first.')

esm = pd.concat(esm_frames, ignore_index=True) if esm_frames else None
if esm is None:
    raise RuntimeError('Need ESM-2 scores first — run notebook 11.')
print(f'ESM-2 scores loaded: {len(esm):,} variants')
print(esm.columns.tolist())

## ٤. حمّل PDBs من AlphaFold DB

لكل UniProt ID فريد، حمّل `AF-{uniprot}-F1-model_v4.pdb`. متوسط حجم ٢٥٠KB/file، لـ~١٤K unique UniProts = ~٣.٥ GB.

In [ ]:
import requests, time, os
from pathlib import Path

AF_URL = 'https://alphafold.ebi.ac.uk/files/AF-{uniprot}-F1-model_v4.pdb'

# Unique UniProt IDs from ESM scores (transcript_id column might actually be
# transcript, not uniprot — depending on how src/esm2_scorer.py stored it).
# Need to fetch UniProt via VEP if missing.

uniprots = set()
if 'uniprot_id' in esm.columns:
    uniprots = set(esm['uniprot_id'].dropna().unique())
else:
    print('⚠ No uniprot_id column in ESM output. Need to fetch via VEP REST.')
    # Quick lookup via VEP REST batched.
    # (Implementation: POST /vep/human/region, take t.get('swissprot').)
    # Skip for now — add in next iteration.

print(f'Unique UniProt IDs to fetch: {len(uniprots):,}')

cached = set(f.stem.replace('AF-', '').replace('-F1-model_v4', '')
             for f in Path(PDB_CACHE).glob('AF-*.pdb'))
to_fetch = uniprots - cached
print(f'  already cached: {len(cached):,}')
print(f'  to fetch:       {len(to_fetch):,}')

for i, uni in enumerate(sorted(to_fetch), 1):
    url = AF_URL.format(uniprot=uni)
    out = Path(PDB_CACHE) / f'AF-{uni}-F1-model_v4.pdb'
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            out.write_text(r.text)
    except requests.RequestException:
        pass
    if i % 100 == 0:
        print(f'  downloaded {i:,}/{len(to_fetch):,}')
    time.sleep(0.1)

print(f'\n✅ PDB cache: {len(list(Path(PDB_CACHE).glob("AF-*.pdb"))):,} files')

## ٥. استخرج structural features

تفاصيل التنفيذ في `src/structural_features.py` (سيُضاف لاحقاً عند الحاجة).

In [ ]:
# Placeholder: the actual extractor will live in src/structural_features.py
# (implemented as part of Stage 2.2 back on the main repo).
# For now, we document the required API here.

print('TODO: src/structural_features.py — compute_all_structural(pdb_path, positions)')
print('Returns DataFrame with: variant_key, pLDDT_position, pLDDT_window_5,')
print('                        SASA_position, secondary_structure, neighbors_5A')

## ٦. Push على GitHub

In [ ]:
%cd /content/repo
!git config user.email 'colab@automation'
!git config user.name 'Colab Automation'
!mkdir -p data/intermediate/alphafold
!cp {FEATURES_DIR}/*.parquet data/intermediate/alphafold/
!git add data/intermediate/alphafold/
!git commit -m 'Stage 2.2 AlphaFold: pLDDT/SASA/depth features (Colab)'
!git push origin HEAD:main